# Generate well-level bulk profiles

This is the final step of the module.
We aggregate the position-corrected single-cell profiles from step 3b to the well level (median per well) and pool the wells of **all plates in all batches**.
The single-cell profiles are already standardized (z-scored using all cells on each plate) and corrected for plate-position effects, so we do not normalize the bulk profiles again.

We then perform one feature selection on the pooled profiles.

**Input** (per plate, from step 3b): `<plate>_sc_position_corrected.parquet`

**Output** (in `data/bulk_profiles/`):

- `bulk_position_corrected_feature_selected.parquet`: pooled well-level profiles of all plates after feature selection

Each well keeps its plate, and we add `Metadata_Batch` and `Metadata_Platemap` (from the folder names) so wells can be grouped by batch and platemap downstream.
Bulk profiles from earlier runs, which were built from uncorrected cells, are not changed.

## Import libraries

In [1]:
import pathlib
import pprint

import pandas as pd
from pycytominer import aggregate, feature_select

## Set paths and variables

In [2]:
# base directory where batches are located
base_dir = pathlib.Path("./data/").resolve(strict=True)

# pooled outputs of all plates
output_dir = base_dir / "bulk_profiles"
output_dir.mkdir(parents=True, exist_ok=True)
output_feature_select_file = (
    output_dir / "bulk_position_corrected_feature_selected.parquet"
)

# operations to perform for feature selection
# pycytominer applies them in this order, so we first drop features with missing
# values and blocklisted features, and then apply the variance and correlation filters
feature_select_ops = [
    "drop_na_columns",
    "blocklist",
    "variance_threshold",
    "correlation_threshold",
]

# columns that identify a well and are constant within it; used as the aggregation
# strata so that the well-level metadata are kept
well_strata = [
    "Metadata_WellRow",
    "Metadata_WellCol",
    "Metadata_heart_number",
    "Metadata_cell_type",
    "Metadata_heart_failure_type",
    "Metadata_treatment",
    "Metadata_Pathway",
    "Metadata_Plate",
    "Metadata_Well",
]

# negative control wells (failing-cell DMSO), counted to check the pooled table
neg_control_query = "Metadata_treatment == 'DMSO' and Metadata_cell_type == 'failing'"

## Set list of plates to process

We process the position-corrected profiles of every plate in every batch.

In [3]:
plate_info_list = [
    {
        "profile_path": profile_path,
        "batch": profile_path.parents[2].name,  # e.g. batch_1
        "platemap": profile_path.parents[1].name,  # e.g. platemap_1
    }
    for profile_path in sorted(
        base_dir.glob(
            "batch_*/platemap_*/single_cell_profiles/*_sc_position_corrected.parquet"
        )
    )
]

# View list
print("Number of plates to process:", len(plate_info_list))
pprint.pprint(plate_info_list[:3], indent=4)

Number of plates to process: 44
[   {   'batch': 'batch_1',
        'platemap': 'platemap_1',
        'profile_path': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles/CARD-CelIns-CX7_251023210001_sc_position_corrected.parquet')},
    {   'batch': 'batch_1',
        'platemap': 'platemap_1',
        'profile_path': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles/CARD-CelIns-CX7_251124150001_sc_position_corrected.parquet')},
    {   'batch': 'batch_1',
        'platemap': 'platemap_1',
        'profile_path': PosixPath('/Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/batch_1/platemap_1/single_cell_profiles/CARD-CelIns-CX7_251125110001_sc_position_corrected.parquet')}]


## Aggregate to the well level and pool all plates

For each plate, we compute the median profile of each well.
QC-failed cells were already removed in step 2 and the profiles are already annotated, so the well-level metadata are kept by using them as the aggregation strata.
We then concatenate the wells of all plates.

In [4]:
well_profiles = []

for plate_info in plate_info_list:
    print("Performing aggregation for", plate_info["profile_path"].name, "...")
    profile_df = pd.read_parquet(plate_info["profile_path"])
    well_df = aggregate(
        population_df=profile_df,
        operation="median",
        strata=well_strata,
    )
    well_df.insert(0, "Metadata_Platemap", plate_info["platemap"])
    well_df.insert(0, "Metadata_Batch", plate_info["batch"])
    well_profiles.append(well_df)

pooled_df = pd.concat(well_profiles, ignore_index=True)
print("Pooled well-level profiles:", pooled_df.shape)
print("Plates:", pooled_df["Metadata_Plate"].nunique())
print("Negative control wells:", len(pooled_df.query(neg_control_query)))

Performing aggregation for CARD-CelIns-CX7_251023210001_sc_position_corrected.parquet ...


Performing aggregation for CARD-CelIns-CX7_251124150001_sc_position_corrected.parquet ...


Performing aggregation for CARD-CelIns-CX7_251125110001_sc_position_corrected.parquet ...


Performing aggregation for CARD-CelIns-CX7_251126130001_sc_position_corrected.parquet ...


Performing aggregation for CARD-CelIns-CX7_251130110002_sc_position_corrected.parquet ...


...

Performing aggregation for CARD-CelIns-CX7_260107160001_sc_position_corrected.parquet ...


Performing aggregation for CARD-CelIns-CX7_260108130001_sc_position_corrected.parquet ...


Performing aggregation for CARD-CelIns-CX7_260109070001_sc_position_corrected.parquet ...


Pooled well-level profiles: (2088, 2464)
Plates: 44


Negative control wells: 176


## Feature selection

We perform one feature selection on the pooled profiles of all plates, so that all wells share one set of features.


In [5]:
print("Feature selecting the pooled profiles...")
feature_select(
    profiles=pooled_df,
    operation=feature_select_ops,
    # drop every feature with a missing value (the pycytominer default allows 5%)
    na_cutoff=0,
    blocklist_file="./blocklist_features.txt",
    # 0.95 is less strict than the pycytominer default of 0.9, so fewer correlated
    # features are removed and more features remain
    corr_threshold=0.95,
    # 0.05 is the pycytominer default for the most common value of a feature
    freq_cut=0.05,
    output_file=output_feature_select_file,
    output_type="parquet",
)

print(f"Saved feature-selected profiles to {output_feature_select_file}")


Feature selecting the pooled profiles...


Saved feature-selected profiles to /Users/waygr/mnt/bandicoot/CFReT_screening_data/screen_profiles_ibp_v2/bulk_profiles/bulk_position_corrected_feature_selected.parquet


In [6]:
# Check an example output file
test_df = pd.read_parquet(output_feature_select_file)

print(test_df.shape)
print("Plate:", test_df.Metadata_Plate.unique())
print(
    "Metadata columns:", [col for col in test_df.columns if col.startswith("Metadata_")]
)
test_df.head(2)

(2088, 1264)
Plate: ['CARD-CelIns-CX7_251023210001' 'CARD-CelIns-CX7_251124150001'
 'CARD-CelIns-CX7_251125110001' 'CARD-CelIns-CX7_251126130001'
 'CARD-CelIns-CX7_251130110002' 'CARD-CelIns-CX7_251201110001'
 'CARD-CelIns-CX7_251201190001' 'CARD-CelIns-CX7_251202100001'
 'CARD-CelIns-CX7_251203170001' 'CARD-CelIns-CX7_251205100001'
 'CARD-CelIns-CX7_251208160001' 'CARD-CelIns-CX7_251210180001'
 'CARD-CelIns-CX7_251211180001' 'CARD-CelIns-CX7_251212100001'
 'CARD-CelIns-CX7_251212180001' 'CARD-CelIns-CX7_251213150001'
 'CARD-CelIns-CX7_251215140002' 'CARD-CelIns-CX7_251216110001'
 'CARD-CelIns-CX7_251216180001' 'CARD-CelIns-CX7_251217120001'
 'CARD-CelIns-CX7_251217190001' 'CARD-CelIns-CX7_251219170001'
 'CARD-CelIns-CX7_251220120001' 'CARD-CelIns-CX7_251229110001'
 'CARD-CelIns-CX7_251230110001' 'CARD-CelIns-CX7_251230180001'
 'CARD-CelIns-CX7_251231100001' 'CARD-CelIns-CX7_260102130001'
 'CARD-CelIns-CX7_260103110001' 'CARD-CelIns-CX7_260103180001'
 'CARD-CelIns-CX7_260104140001' 'CA

,Metadata_Batch,Metadata_Platemap,Metadata_WellRow,Metadata_WellCol,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Pathway,Metadata_Plate,...,Nuclei_Texture_InfoMeas2_PM_3_00_256,Nuclei_Texture_InverseDifferenceMoment_Actin_3_00_256,Nuclei_Texture_InverseDifferenceMoment_DNA_3_02_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_02_256,Nuclei_Texture_InverseDifferenceMoment_Mito_3_00_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_01_256,Nuclei_Texture_SumAverage_Actin_3_03_256,Nuclei_Texture_SumEntropy_Mito_3_00_256,Nuclei_Texture_SumVariance_ER_3_03_256,Nuclei_Texture_SumVariance_Mito_3_01_256
0,batch_1,platemap_1,B,2,7,healthy,None,DMSO,None,CARD-CelIns-CX7_251023210001,...,-0.177440,1.336756,0.080384,-0.030549,0.691689,1.103459,-1.585013,-0.731211,-0.199479,-0.235804
1,batch_1,platemap_1,B,3,25,failing,dilated_cardiomyopathy,UCD-0159256,Apoptosis,CARD-CelIns-CX7_251023210001,...,0.024534,0.443173,0.344346,0.617743,-0.119059,0.446571,-0.628411,0.179554,-0.252208,-0.183406
